In [2]:
import os
from dotenv import load_dotenv

In [3]:
load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Load a Text File with TextLoader

In [4]:
from langchain_community.document_loaders import TextLoader
loader = TextLoader("speech.txt")
text_documents = loader.load()
text_documents

[Document(metadata={'source': 'speech.txt'}, page_content='Good evening everyone.\n\nToday, I want to speak about something that is shaping the future of our worldâ€”our ability to learn, adapt, and reinvent ourselves. We are living in an era where technology evolves faster than our understanding, and every new breakthrough challenges the way we think, work, and interact.\n\nArtificial intelligence, machine learning, and automation are not just tools. They are the new driving forces of global innovation. But the true power of these technologies does not lie in the algorithms alone. It lies in the people who build them, the people who apply them, and the people who believe they can use them to transform lives.\n\nMany times, we hesitate to take the next step because we fear failure. We worry that we are not ready. But the truth is, nobody ever feels fully ready. Growth happens not when we are comfortable, but when we challenge ourselves to move beyond the familiar.\n\nEach of us carries

# Web Page Loader (WebBaseLoader + BeautifulSoup Strainer)

In [6]:
from langchain_community.document_loaders import WebBaseLoader
import bs4

# Load, parse, and extract specific HTML tags from a webpage
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-title", "post_content", "post_header")
        )
    )
)

text_documents = loader.load()

print(f"Loaded {len(text_documents)} HTML document(s)")


Loaded 1 HTML document(s)


# PDF Loader (PyPDFLoader)

In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("attention.pdf")
docs = loader.load()

print(f"Loaded {len(docs)} PDF page(s)")


Loaded 15 PDF page(s)


# Chunk the Documents – RecursiveCharacterTextSplitter

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

documents = text_splitter.split_documents(docs)

# Inspect first 5 chunks
documents[:5]


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszk

# Create Chroma Vector Database (OpenAI Embeddings)

In [13]:
from langchain_community.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings()

# Use first 15 chunks
db = Chroma.from_documents(documents[:15], embeddings)


APIConnectionError: Connection error.

In [17]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# This is a small, very popular model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

db = Chroma.from_documents(documents[:15], embeddings)


c:\Users\samee\OneDrive\Desktop\git\sameerkhan\langchain\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\samee\OneDrive\Desktop\git\sameerkhan\langchain\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\samee\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or t

# Similarity Search on Chroma

In [18]:
query = "Who are the authors of the Attention is All You Need research paper?"
result = db.similarity_search(query)

result


[Document(metadata={'creator': 'LaTeX with hyperref', 'subject': '', 'source': 'attention.pdf', 'trapped': '/False', 'moddate': '2024-04-10T21:11:43+00:00', 'page': 2, 'keywords': '', 'title': '', 'creationdate': '2024-04-10T21:11:43+00:00', 'producer': 'pdfTeX-1.40.25', 'author': '', 'total_pages': 15, 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'page_label': '3'}, page_content='3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3'),
 Document(metadata={'title': '', 'subject': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'author': '', 'producer': 'pdfTeX-1.40.25', 'creationdate': '2024-04-10T21:11:43+00:00', 'trapped': '/False', 'keywords': '', 'creator': 'LaTeX with hyperref', 'moddate': '

# FAISS Vector Database

In [20]:
from langchain_community.vectorstores import FAISS

db1 = FAISS.from_documents(documents[:20], embeddings)

# Optional: search on FAISS
result_faiss = db1.similarity_search(query)

result_faiss


[Document(id='f8dc61b3-efa4-4ff6-97e6-6055879e09de', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 2, 'page_label': '3'}, page_content='3.2 Attention\nAn attention function can be described as mapping a query and a set of key-value pairs to an output,\nwhere the query, keys, values, and output are all vectors. The output is computed as a weighted sum\n3'),
 Document(id='4e5ea0cf-bac8-4853-8185-0a5e4fb42a4b', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 

In [22]:
query = "Embeddings and Softmax"
result = db1.similarity_search(query)
result[0].page_content

'FFN(x) = max(0, xW1 + b1)W2 + b2 (2)\nWhile the linear transformations are the same across different positions, they use different parameters\nfrom layer to layer. Another way of describing this is as two convolutions with kernel size 1.\nThe dimensionality of input and output is dmodel = 512, and the inner-layer has dimensionality\ndff = 2048.\n3.4 Embeddings and Softmax\nSimilarly to other sequence transduction models, we use learned embeddings to convert the input\ntokens and output tokens to vectors of dimension dmodel. We also use the usual learned linear transfor-\nmation and softmax function to convert the decoder output to predicted next-token probabilities. In\nour model, we share the same weight matrix between the two embedding layers and the pre-softmax\nlinear transformation, similar to [30]. In the embedding layers, we multiply those weights by √dmodel.\n5'